In [1]:

from mri_loader import Subject
from nilearn.glm.first_level import FirstLevelModel

import pandas as pd
import numpy as np

from stats import *




In [2]:
subject_ids = [2, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 21, 23]

run_ids = list(range(1,5))

(subject_ids, run_ids)

([2, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
 [1, 2, 3, 4])

In [3]:

global_z_map = {}

contrast_list = [{"+": ["high"], "-": ["low"]},  # high > low
                 {"+": ["undecided"], "-": ["high", "low"]},]  # undecided > high + low

scale = [["35", "45"], ["45", "55"], ["55", "65"], ["65", "75"], ["75", "85"], ["85", "95"]]
to_subtract = {"-": ["5", "15"]}

for values in scale:
    contrast_list.append({
        "+": values,
        **to_subtract
    })

for classes in contrast_list:
    name = contrast_name(classes)
    global_z_map[name] = []

global_z_map

{'high > low': [],
 'undecided > high + low': [],
 '35 + 45 > 5 + 15': [],
 '45 + 55 > 5 + 15': [],
 '55 + 65 > 5 + 15': [],
 '65 + 75 > 5 + 15': [],
 '75 + 85 > 5 + 15': [],
 '85 + 95 > 5 + 15': []}

In [4]:


for subject in subject_ids:

    try:
        dataset = Subject(subject, run_ids)
        dataset.load()

        images, times, labels = dataset.get_data()
        low_inflexion, high_inflexion = dataset.compute_inflexions()
    except Exception as e:
        print("Skipping subject ", subject)
        print(e)
        continue

    print(f"{subject=} {low_inflexion=}, {high_inflexion=}")

    labels_class = set(labels)

    events = pd.DataFrame(
        {'onset': times,
         'trial_type': labels,
         'duration': 2.5}
    )

    repetition_time = dataset.repetition_time
    fmri_glm = FirstLevelModel(t_r=repetition_time,
              drift_model='polynomial',
              drift_order=3,
              hrf_model='spm',
              mask_img=dataset.brain_mask,
              smoothing_fwhm=6,
              n_jobs=-1)

    fmri_glm = fmri_glm.fit(images, events)

    design_matrix = fmri_glm.design_matrices_[0]

    contrast_matrix = np.eye(design_matrix.shape[1])
    contrasts = {
        str(column): contrast_matrix[i]
        for i, column in enumerate(design_matrix.columns)
    }

    low_contrast_columns = []
    high_contrast_columns = []
    undecided_contrast_columns = []

    for key, column in contrasts.items():
        try:
            key_numeric = float(key) / 100

            if key_numeric < low_inflexion:
                low_contrast_columns.append(column)

            elif key_numeric > high_inflexion:
                high_contrast_columns.append(column)

            else:
                undecided_contrast_columns.append(column)

        except ValueError:
            continue

    contrasts["low"]       = np.sum(low_contrast_columns, axis=0)
    contrasts["high"]      = np.sum(high_contrast_columns, axis=0)
    contrasts["undecided"] = np.sum(undecided_contrast_columns, axis=0)

    for contrast in contrast_list:

        glm_contrast_vector  = np.sum(contrasts[column] for column in contrast["+"])
        glm_contrast_vector -= np.sum(contrasts[column] for column in contrast["-"])

        z_score = fmri_glm.compute_contrast(glm_contrast_vector, output_type="z_score")
        name = contrast_name(contrast)

        global_z_map[name].append(z_score)




subject=2 low_inflexion=np.float64(0.2698198198198198), high_inflexion=np.float64(0.613063063063063)
subject=6 low_inflexion=np.float64(0.2734234234234234), high_inflexion=np.float64(0.7103603603603603)
subject=7 low_inflexion=np.float64(0.1725225225225225), high_inflexion=np.float64(0.7238738738738738)
subject=8 low_inflexion=np.float64(0.07072072072072072), high_inflexion=np.float64(0.7247747747747747)
subject=9 low_inflexion=np.float64(0.2806306306306306), high_inflexion=np.float64(0.3842342342342342)
subject=10 low_inflexion=np.float64(0.07072072072072072), high_inflexion=np.float64(0.7490990990990991)
subject=11 low_inflexion=np.float64(0.2905405405405405), high_inflexion=np.float64(0.7076576576576576)
subject=12 low_inflexion=np.float64(0.06171171171171171), high_inflexion=np.float64(0.6490990990990991)
subject=14 low_inflexion=np.float64(0.20045045045045046), high_inflexion=np.float64(0.7977477477477477)
subject=15 low_inflexion=np.float64(0.25360360360360357), high_inflexion=np

In [8]:
from nilearn.reporting import get_clusters_table
from nilearn.glm import threshold_stats_img
from brain_map import find_region_names

from nilearn import datasets


In [10]:

atlas_name = 'cort-maxprob-thr0-1mm'
harvard = datasets.fetch_atlas_harvard_oxford(atlas_name=atlas_name)

atlas_img = harvard.maps
labels = harvard.labels

regions_activated = {}

for contrast_name, images in global_z_map.items():
    regions_activated[contrast_name] = []

    for z_score in images:

        clean, threshold = threshold_stats_img(z_score, alpha=0.001, height_control="fpr", cluster_threshold=5)
        table = get_clusters_table(
            clean, stat_threshold=threshold, cluster_threshold=5
        )

        pos = [(x,y,z) for (x,y,z) in zip(table['X'], table['Z'], table['Y'])]

        try:
            indexes, regions_names = find_region_names(pos, atlas_img, labels=labels)
        except Exception as e:
            print(e)
            continue

        regions_names = list(set(str(v) for v in regions_names))

        regions_activated[contrast_name].append(regions_names)

        print(f"{contrast_name=}, {regions_names=}")


[fetch_atlas_harvard_oxford] Dataset found in C:\Users\ducat\nilearn_data\fsl

contrast_name='high > low', regions_names=['Temporal Pole', 'Insular Cortex', 'Inferior Frontal Gyrus, pars opercularis', 'Background', 'Middle Frontal Gyrus', 'Planum Temporale', 'Frontal Orbital Cortex', 'Frontal Pole', 'Superior Frontal Gyrus']
contrast_name='high > low', regions_names=['Central Opercular Cortex', 'Inferior Frontal Gyrus, pars opercularis', 'Supramarginal Gyrus, anterior division', 'Background', 'Middle Frontal Gyrus', 'Precentral Gyrus', 'Subcallosal Cortex', 'Frontal Pole', 'Postcentral Gyrus', 'Frontal Orbital Cortex', 'Middle Temporal Gyrus, posterior division', 'Parietal Opercular Cortex', 'Inferior Frontal Gyrus, pars triangularis', 'Superior Temporal Gyrus, posterior division', 'Inferior Temporal Gyrus, anterior division', 'Middle Temporal Gyrus, anterior division', 'Superior Frontal Gyrus', 'Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)']
contrast_name='high > low', regions_names=['Temporal Pole', 'Background', 'Middle Frontal Gyrus', '

In [12]:
import pandas as pd



In [35]:

regions_activated_np = {}
regions_count = {}

for k, v in regions_activated.items():
    regions_activated_np[k] = np.array(v, dtype=object)

    regions_activated_np[k] = np.concatenate(regions_activated[k])

    regions_count[k] = np.unique_counts(regions_activated_np[k])

regions_count

{'high > low': UniqueCountsResult(values=array(['Background', 'Central Opercular Cortex',
        'Cingulate Gyrus, anterior division', 'Frontal Medial Cortex',
        'Frontal Opercular Cortex', 'Frontal Orbital Cortex',
        'Frontal Pole', "Heschl's Gyrus (includes H1 and H2)",
        'Inferior Frontal Gyrus, pars opercularis',
        'Inferior Frontal Gyrus, pars triangularis',
        'Inferior Temporal Gyrus, anterior division',
        'Inferior Temporal Gyrus, posterior division', 'Insular Cortex',
        'Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex)',
        'Middle Frontal Gyrus', 'Middle Temporal Gyrus, anterior division',
        'Middle Temporal Gyrus, posterior division',
        'Middle Temporal Gyrus, temporooccipital part',
        'Paracingulate Gyrus', 'Parahippocampal Gyrus, anterior division',
        'Parietal Opercular Cortex', 'Planum Polare', 'Planum Temporale',
        'Postcentral Gyrus', 'Precentral Gyrus', 'Subcallosal Cortex'

In [37]:

df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in regions_activated_np.items()]))

for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(6))


--- high > low ---
high > low
Background                14
Frontal Orbital Cortex    12
Frontal Pole              11
Superior Frontal Gyrus     9
Middle Frontal Gyrus       9
Precentral Gyrus           8
Name: count, dtype: int64

--- undecided > high + low ---
undecided > high + low
Background                13
Frontal Pole              11
Middle Frontal Gyrus      10
Superior Frontal Gyrus     9
Precentral Gyrus           9
Paracingulate Gyrus        7
Name: count, dtype: int64

--- 35 + 45 > 5 + 15 ---
35 + 45 > 5 + 15
Background                14
Middle Frontal Gyrus      13
Frontal Pole              13
Temporal Pole             11
Precentral Gyrus          11
Superior Frontal Gyrus    11
Name: count, dtype: int64

--- 45 + 55 > 5 + 15 ---
45 + 55 > 5 + 15
Background              14
Frontal Pole            14
Temporal Pole           11
Precentral Gyrus        10
Middle Frontal Gyrus     9
Paracingulate Gyrus      9
Name: count, dtype: int64

--- 55 + 65 > 5 + 15 ---
55 + 65 > 5 + 

In [44]:
summary = pd.DataFrame({
    col: df[col].value_counts().head(8)
    for col in df.columns
})

summary

,high > low,undecided > high + low,35 + 45 > 5 + 15,45 + 55 > 5 + 15,55 + 65 > 5 + 15,65 + 75 > 5 + 15,75 + 85 > 5 + 15,85 + 95 > 5 + 15
Background,14.0,13.0,14.0,14.0,14.0,14.0,14.0,14.0
"Cingulate Gyrus, anterior division",NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN
Frontal Orbital Cortex,12.0,NaN,11.0,NaN,8.0,9.0,10.0,10.0
Frontal Pole,11.0,11.0,13.0,14.0,14.0,14.0,14.0,14.0
"Inferior Frontal Gyrus, pars triangularis",NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN
Insular Cortex,NaN,NaN,NaN,8.0,NaN,10.0,7.0,NaN
Middle Frontal Gyrus,9.0,10.0,13.0,9.0,12.0,9.0,11.0,11.0
Paracingulate Gyrus,NaN,7.0,9.0,9.0,9.0,NaN,NaN,8.0
Precentral Gyrus,8.0,9.0,11.0,10.0,12.0,8.0,10.0,9.0
Subcallosal Cortex,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
